# 07 — BAF LTN and Explanation Generalization

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

In [ ]:
from src.experiment import run_predictive_benchmarks
from src.explanation import RuleExplainer, bootstrap_explanation_precision_gain, explanation_quality_metrics, rule_quality_table
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "06_baf_generalization"
result = run_predictive_benchmarks(
    PROJECT_ROOT / "configs/baf.yaml",
    output_dir=output_dir,
    model_names=("mlp", "tabular_resnet", "tree"),
    quick_run=QUICK_RUN,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
)
config = result["config"]
prepared = result["prepared"]
display(result["metrics"].round(4))
print({"data_source": result["data_source"], "rows": len(result["frame"])})

## Data

In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(prepared.y_train), len(prepared.y_validation), len(prepared.y_test)],
    "fraud_rate": [prepared.y_train.mean(), prepared.y_validation.mean(), prepared.y_test.mean()],
})
display(split_summary)

## Results

In [ ]:
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
test_truth = engine.evaluate(prepared.test_frame)
rule_quality = rule_quality_table(test_truth, prepared.y_test, config["logic"]["activation_threshold"])
display(rule_quality.round(4))

model_key = "tree"
probabilities = result["test_probabilities"][model_key]
threshold = result["thresholds"][model_key]
explainer = RuleExplainer(engine, config["logic"]["activation_threshold"], config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations,
    prepared.y_test,
    probabilities,
    threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"],
    seed=config["project"]["seed"],
))
display(pd.DataFrame([quality]).round(4))

rule_quality.to_csv(output_dir / "baf_rule_quality.csv", index=False)
pd.DataFrame([quality]).to_csv(output_dir / "baf_explanation_quality.csv", index=False)

In [ ]:
test_metrics = result["metrics"].query("split == 'test'")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=test_metrics, x="model", y="pr_auc", ax=axes[0], color="#4C72B0")
axes[0].set_title("BAF test PR-AUC")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=rule_quality, x="rule", y="lift", ax=axes[1], color="#C44E52")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("BAF rule lift")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(output_dir / "baf_generalization.png", dpi=160, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
best_model = test_metrics.sort_values("pr_auc", ascending=False).iloc[0]
best_rule = rule_quality.sort_values("lift", ascending=False).iloc[0]
display(Markdown(
    f"- Data source: **{result['data_source']}**.\n"
    f"- Highest BAF test PR-AUC: **{best_model['model']} = {best_model['pr_auc']:.4f}**.\n"
    f"- Highest BAF rule lift: **{best_rule['rule']} = {best_rule['lift']:.3f}**.\n"
    f"- Tree-alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**.\n"
    "- Cross-dataset comparison is valid only for real-data full runs."
))